# Exploratory Data Analysis (EDA)

**Notebook Objectives:**

This notebook performs the initial exploratory data analysis (EDA) for the Banking Fraud Detection project.

**The goals are:**

1. Load and inspect the raw dataset
2. Understand dataset structure and feature types
3. Analyze fraud class imbalance
4. Detect missing values and duplicates
5. Explore feature distributions
6. Analyze correlations between variables
7. Identify potential fraud indicators
8. Generate visualizations for reporting
9. Prepare insights for feature engineering

# 1. Imports & Setup

We first import all required libraries and project modules.

**This notebook uses:**

* pandas for data analysis
* matplotlib and seaborn for visualization
* reusable loading utilities from src/

In [ ]:
# Setup to allow importing from src
import sys
sys.path.append("..")

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt 
import seaborn as sns

# Project Modules
from src.config import (
    REPORT_DIR,
    TARGET_COLUMN
)

from src.data_loader import (
    load_raw_data,
    summarize_dataset,
    check_duplicates
)

# Plot Styling
sns.set_style("whitegrid")

# Figures Directory
FIGURES_DIR = REPORT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Setup complete. Figures will be automatically saved to: {FIGURES_DIR}")

# 2. Load Dataset

We now load the raw banking transaction dataset using our reusable data loader utility.

In [ ]:
df = load_raw_data()

# 3. Dataset Overview

**We inspect:**

* dataset dimensions
* column names
* data types
* missing values
* duplicates

In [ ]:
summarize_dataset(df)

In [ ]:
check_duplicates(df)

In [ ]:
df.head()

# 4. Statistical Summary

We examine descriptive statistics for numerical variables.

**This helps identify:**

* outliers
* unusual ranges
* scaling issues
* suspicious distributions

In [ ]:
df.describe().T

# 5. Fraud Class Distribution

Fraud datasets are usually highly imbalanced.

Understanding the target distribution is critical before modeling.

In [ ]:
df[TARGET_COLUMN].value_counts()

In [ ]:
# Plot Distribution
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x=TARGET_COLUMN,
    hue=TARGET_COLUMN,
    palette="viridis"
)

plt.title("Fraud Class Distribution")

# Save Figure
plt.savefig(
    FIGURES_DIR / "fraud_class_distribution.png",
    bbox_inches="tight",
    dpi=300
)
plt.show()

# 7. Numerical Feature Distributions

**We visualize important numerical variables to understand:**

* skewness
* spread
* abnormal patterns
* potential fraud behavior

In [ ]:
numerical_columns = df.select_dtypes(
    include=np.number
).drop(columns=["transaction_id"], errors="ignore").columns.to_list()

numerical_columns

In [ ]:
# Plots Distribution
for column in numerical_columns:
    plt.figure(figsize=(8, 4))
    
    sns.histplot(
        df[column],
        bins=30,
        kde=True
    )
    
    plt.title(f"Distribution of: {column}")
    
    # Save Plots
    plt.savefig(
        FIGURES_DIR / f"{column}_distribution.png",
        bbox_inches="tight",
        dpi=300
    )
    
    plt.show()

# 8. Correlation Analysis

We compute correlations between numerical variables.

**Correlation analysis helps identify:**

* multicollinearity
* redundant variables
* fraud-related relationships

In [ ]:
correlation_matrix = df[numerical_columns].corr()
correlation_matrix.head()

In [ ]:
# Plot Correlation Matrix Heatmap
plt.figure(figsize=(8, 4))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    annot=False,
    fmt=".2f",
)

plt.title("Feature Correlation Heatmap")

# Saving Figure
plt.savefig(
    FIGURES_DIR / "correlation_heatmap.png",
    bbox_inches="tight",
    dpi=300
)
plt.show()

# Correlation Matrix Interpretation

## Objective

The correlation matrix helps us understand the linear relationships between numerical features in the banking fraud dataset.

Correlation values range between:

- `+1` → strong positive correlation
- `0` → no linear relationship
- `-1` → strong negative correlation

---

# Key Observation

Most correlations in this dataset are relatively weak (close to 0).

This is actually common in fraud detection datasets because fraudulent behavior is usually:
- nonlinear
- complex
- interaction-based
- pattern-driven rather than single-feature driven

This suggests that:
- simple linear relationships are limited
- ensemble/tree-based models may perform better
- feature engineering will be important

---

# Notable Findings

## 1. Low Multicollinearity

Most features show very weak correlations with each other.

This is beneficial because:
- models are less affected by redundant information
- features provide more independent signals
- linear models become more stable

Implication:
- Logistic Regression can perform reliably
- Feature redundancy is minimal
- Dimensionality reduction may not be necessary

---

## 2. Transaction Behavior Features

Features such as:
- `transaction_velocity_score`
- `transfer_frequency`
- `daily_transaction_count`

are expected to contribute to fraud risk even if pairwise correlations are weak.

Fraud patterns are often detected through:
- combinations of variables
- thresholds
- unusual behavioral interactions

---

## 3. Risk Indicators

Important fraud-related indicators include:
- `anomaly_score`
- `device_risk_score`
- `suspicious_ip_flag`
- `international_transaction_flag`

Even without strong direct correlations, these variables may still become highly important in:
- Random Forest
- Gradient Boosting
- XGBoost
- Isolation Forest

because tree-based models capture nonlinear interactions.

---

## 4. No Severe Correlation Problems

No extremely high correlations (e.g. > 0.90) were observed.

This means:
- multicollinearity risk is low
- feature leakage is unlikely
- the dataset is structurally healthy for ML training

---

# Important ML Implications

## Linear Models

Models like:
- Logistic Regression
- Linear SVM

may struggle to capture complex fraud behavior unless strong feature engineering is added.

---

## Tree-Based Models

Models like:
- Decision Trees
- Random Forest
- Gradient Boosting

will likely perform better because they:
- capture nonlinear interactions
- model thresholds naturally
- handle fraud behavior patterns effectively

---

## Need for Feature Engineering

Since raw correlations are weak, engineered features may significantly improve performance.

Potential examples:
- transaction ratios
- velocity aggregation
- risk combinations
- behavioral scoring systems

---

# Final EDA Conclusion

The correlation analysis suggests that fraud detection in this dataset is not driven by simple linear relationships.

Instead, fraud appears to emerge from:
- behavioral patterns
- interaction effects
- anomaly combinations
- transaction context

This reinforces the importance of:
- feature engineering
- ensemble learning
- anomaly detection methods
- imbalance handling techniques

# 9. Fraud vs Non-Fraud Feature Comparison

We compare feature distributions across fraud classes.

This helps identify discriminative features.

In [ ]:
important_features = [
    "transaction_amount",
    "anomaly_score",
    "device_risk_score",
    "transfer_frequency",
    "login_attempts"
]

In [ ]:
# Plotting important features vs Fraud
for feature in important_features:
    plt.figure(figsize=(8, 4))
    
    sns.boxplot(
        data=df,
        x=TARGET_COLUMN,
        y=feature
    )
    
    plt.title(f"{feature} by Fraud Class")
    
    # Saving Plots
    plt.savefig(
        FIGURES_DIR / f"{feature}_fraud_comparison.png",
        bbox_inches="tight",
        dpi=300
    )
    
    plt.show()

# 10. Outlier Analysis

Fraudulent transactions often appear as outliers.

We inspect outliers visually using boxplots.

In [ ]:
# Plotting Outliers
for feature in important_features:
    plt.figure(figsize=(8, 4))
    
    sns.boxplot(
        x=df[feature]
    )
    plt.title(f"Outliers Detection of: {feature}")
    
    # Saving Plots
    plt.savefig(
        FIGURES_DIR / f"{feature}_outliers.png",
        bbox_inches="tight",
        dpi=300
    )
    
    plt.show()